In [1]:
#Face enrolment: 
import cv2
import urllib.request
import numpy as np
import os  # For handling directory creation

# Load the Haar cascades for face and eye detection
f_cas = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

# URL for your ESP32 camera feed
url = 'http://172.20.10.6/320x240.jpg'

# Set up the window
cv2.namedWindow("Live Transmission", cv2.WINDOW_AUTOSIZE)

# Face ID counter
face_id = 0

# Dictionary to store face IDs and corresponding labels
face_labels = {}

# Directory to save enrolled face images
enrollment_dir = "enrolled_faces"

# Create the directory if it doesn't exist
if not os.path.exists(enrollment_dir):
    os.makedirs(enrollment_dir)

# Function to check if two rectangles intersect
def rectangles_intersect(rect1, rect2):
    x1, y1, w1, h1 = rect1
    x2, y2, w2, h2 = rect2
    # Check if the rectangles intersect by comparing coordinates
    return not (x1 + w1 <= x2 or x2 + w2 <= x1 or y1 + h1 <= y2 or y2 + h2 <= y1)

while True:
    # Fetch the image from ESP32 camera
    img_resp = urllib.request.urlopen(url)
    imgnp = np.array(bytearray(img_resp.read()), dtype=np.uint8)
    img = cv2.imdecode(imgnp, -1)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Detect faces in the image
    faces = f_cas.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    
    for x, y, w, h in faces:
        # Draw a rectangle around the face
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 0, 255), 3)
        roi_gray = gray[y:y + h, x:x + w]
        roi_color = img[y:y + h, x:x + w]
        
        # Detect eyes in the face region
        eyes = eye_cascade.detectMultiScale(roi_gray)
        
        # Ensure both eyes are detected (at least 2 eyes), and they do not intersect
        valid_eyes = []
        for i, (ex1, ey1, ew1, eh1) in enumerate(eyes):
            for j, (ex2, ey2, ew2, eh2) in enumerate(eyes):
                if i < j:  # Only compare unique pairs of eyes
                    if not rectangles_intersect((ex1, ey1, ew1, eh1), (ex2, ey2, ew2, eh2)):
                        valid_eyes.append(((ex1, ey1, ew1, eh1), (ex2, ey2, ew2, eh2)))
        
        # If a valid pair of eyes is found, proceed with enrollment
        if valid_eyes:
            # Take the first valid pair of eyes
            eye1, eye2 = valid_eyes[0]
            (ex1, ey1, ew1, eh1) = eye1
            (ex2, ey2, ew2, eh2) = eye2
            
            # Draw rectangles around detected eyes
            cv2.rectangle(roi_color, (ex1, ey1), (ex1 + ew1, ey1 + eh1), (0, 255, 0), 2)
            cv2.rectangle(roi_color, (ex2, ey2), (ex2 + ew2, ey2 + eh2), (0, 255, 0), 2)
            
            # Prompt the user to input a label (e.g., person's name or ID)
            label = input(f"Enter label for face {face_id} (e.g., person's name or ID): ")
            
            # Save the face image with the label in the "enrolled_faces" directory
            face_filename = os.path.join(enrollment_dir, f"face_{face_id}_{label}.jpg")  # Include label in the filename
            cv2.imwrite(face_filename, roi_color)  # Save the detected face image
            print(f"Face saved as {face_filename}")
            
            # Store the label for this face in the dictionary
            face_labels[face_id] = label
            
            # Increment face ID for next enrollment
            face_id += 1
        else:
            print("Eyes detected are intersecting. Face enrollment not allowed.")

    # Show the live video stream
    cv2.imshow("Live Transmission", img)

    # Wait for the user to press 'q' to quit
    key = cv2.waitKey(5)
    if key == ord('q'):
        break

# Release resources and close windows
cv2.destroyAllWindows()

# After enrollment, print the labels for each face
print("Face labels:")
for face_id, label in face_labels.items():
    print(f"Face ID: {face_id}, Label: {label}")

/opt/anaconda3/lib/python3.12/site-packages/IPython/core/inputtransformer2.py:627: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens_by_line = make_tokens_by_line(lines)


Enter label for face 0 (e.g., person's name or ID):  Samuel


Face saved as enrolled_faces/face_0_Samuel.jpg


Enter label for face 1 (e.g., person's name or ID):  Samuel


Face saved as enrolled_faces/face_1_Samuel.jpg


...

Enter label for face 8 (e.g., person's name or ID):  Jiang Yi


Face saved as enrolled_faces/face_8_Jiang Yi.jpg
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face

Enter label for face 9 (e.g., person's name or ID):  Jiang Yi


Face saved as enrolled_faces/face_9_Jiang Yi.jpg
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face enrollment not allowed.
Eyes detected are intersecting. Face

KeyboardInterrupt: 